# Data Preparation and Loading
This section handles the loading and preprocessing of the TAPER dataset for tactile gesture recognition.
The dataset consists of time-series signals from sensors mounted on gloves, capturing different types of taps.
Key preprocessing steps include:
- Loading pre-processed CSV files containing feature vectors and labels
- Handling different sampling rates to study temporal resolution effects
- Splitting data into training features and labels for model evaluation

In [1]:
%pip install pandas matplotlib numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

import numpy as np

# Data Loading and Preprocessing for TAPER Dataset
# The TAPER dataset consists of time-series signals captured from IMU sensor mounted on the finger.
# Signals are pre-processed and stored in CSV format for different mounting technologies (rings, tape_thick) and sampling rates.
# Each row in the CSV represents a single tap event, with the first column containing the class label (0-3 for 4 classes)
# and subsequent columns containing the feature vector (time-series data points).
# This function loads the specified dataset, handles different sampling rates, and splits into features (X) and labels (y).

def get_data(mounting_tech = 'rings', sampling_rate = 32):
    # Determine file suffix based on sampling rate
    # For 32kHz, the suffix is '32'; for other rates, use the rate value directly
    if sampling_rate == 32 or sampling_rate == 32.0:
        path_suffix = '32'
    else:
        path_suffix = f'{sampling_rate}'

    # Helper function to load CSV data with error handling for empty files
    def load_csv_with_minimum_columns(filepath):
        file_data = np.loadtxt(filepath, delimiter=',', ndmin=2)
        if file_data.size == 0:
            raise ValueError(f"Empty CSV file: {filepath}")
        
        return file_data
    
    # Load the pre-processed CSV file for the specified mounting technology and sampling rate
    file_data = load_csv_with_minimum_columns(
        f'TAPER_FULL_DATASET/pre-processed/{mounting_tech}/{mounting_tech}_{path_suffix}khz.csv'
    )

    # Concatenate data if multiple files were loaded (currently only one file per configuration)
    data_list = [file_data]
    data = np.concatenate(data_list, axis=0)

    # Output dataset statistics for verification
    print("loaded data shape:", data.shape)
    print("number of taps:", len(data))
    
    # Split data into features (X) and labels (y)
    # Labels are in column 0, features in columns 1:end
    data_X = []
    data_y = [] 
    for i in range(len(data)):
        data_X.append(data[i][1:])
        data_y.append(data[i][0])
        
    data_X = np.array(data_X)
    data_y = np.array(data_y)

    return data, data_X, data_y

# Load dataset for evaluation
# Mounting technology options: 'rings' (ring-mounted sensors), 'tape_thick' (tape-mounted sensors)
mounting_tech = 'rings'  # 'rings', 'tape_thick', or 'both'
data, data_X, data_y = get_data(mounting_tech=mounting_tech)

loaded data shape: (3166, 4696)
number of taps: 3166


# Model Evaluations
This section evaluates multiple deep learning architectures for tactile gesture classification:
- Multi-Layer Perceptron (MLP): Fully-connected network baseline
- CNN-BiLSTM: Convolutional feature extraction + bidirectional temporal modeling
- Fully Convolutional Network (FCN): Efficient conv-only architecture
- Residual Network (ResNet): Deep network with skip connections
- SVM: Traditional machine learning baseline
- Random Forest: Ensemble tree-based baseline
All models are evaluated using 5-fold stratified cross-validation across multiple sampling rates
to assess robustness to temporal resolution and computational efficiency trade-offs.

In [ ]:
# Cross-Validation Setup
# To ensure robust evaluation and mitigate overfitting, we employ stratified k-fold cross-validation.
# Stratified sampling maintains the class distribution in each fold, which is crucial for imbalanced datasets.
# We use 5 folds as a standard choice balancing computational cost and statistical reliability.
# Random state is fixed for reproducibility of results across runs.
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

In [ ]:
# Select mounting technology for evaluation
# This variable controls which sensor mounting configuration to evaluate.
# 'rings': Ring-mounted IMU sensor
# 'tape_thick': Tape-mounted sensor with thick adhesive
# The choice affects the data loading and may influence model performance due to different sensor characteristics.
mounting_tech = 'tape_thick'  # 'rings' or 'tape_thick'

## MLP

In [ ]:
# Multi-Layer Perceptron (MLP) Architecture
# MLPs are fully-connected neural networks suitable for tabular or flattened time-series data.
# We implement a deep network with multiple hidden layers to capture complex patterns in tactile signals.
# Architecture choices:
# - Input dropout (0.1): Reduces overfitting by randomly dropping input features
# - Three hidden layers with 500 units each: Provides sufficient capacity for the task
# - ReLU activation: Standard choice for hidden layers, helps with vanishing gradients
# - Progressive dropout (0.2, 0.2, 0.3): Increasing dropout rates to regularize deeper layers
# - Output layer: 4 units with softmax for multi-class classification (4 tap types)
# - Optimizer: Adadelta - Adaptive learning rate optimizer that adjusts based on past gradients
# - Loss: Sparse categorical crossentropy - Appropriate for integer-encoded class labels
# This architecture follows best practices for deep MLPs in classification tasks.

import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, Flatten, LSTM, Dense, Bidirectional, Dropout)
import numpy as np

def build_mlp(input_shape):
    inputs = keras.layers.Input(input_shape)
    x = keras.layers.Dropout(0.1)(inputs)  # Input regularization
    x = keras.layers.Dense(500, activation='relu')(x)  # First hidden layer
    x = keras.layers.Dropout(0.2)(x)  # Dropout for regularization
    x = keras.layers.Dense(500, activation='relu')(x)  # Second hidden layer
    x = keras.layers.Dropout(0.2)(x)  # Dropout for regularization
    x = keras.layers.Dense(500, activation = 'relu')(x)  # Third hidden layer
    x = keras.layers.Dropout(0.3)(x)  # Higher dropout for deeper layer
    out = keras.layers.Dense(4, activation='softmax')(x)  # Output layer for 4 classes
    
    model = keras.models.Model(inputs=inputs, outputs=out)
    optimizer = keras.optimizers.Adadelta()  # Adaptive optimizer
    model.compile(loss='sparse_categorical_crossentropy',
                    optimizer=optimizer,
                    metrics=['accuracy'])  
    
    return model

In [ ]:
# MLP Model Evaluation Across Sampling Rates
# To assess the impact of temporal resolution on classification performance, we evaluate the MLP
# across multiple downsampling factors. Lower sampling rates reduce computational cost but may
# lose high-frequency information critical for distinguishing tap types.
# Sampling rates tested: 32kHz down to 0.01kHz (factors 1 to 3200)
# For each sampling rate, we perform 5-fold cross-validation to obtain reliable performance estimates.

for i in [1, 2, 4, 8, 16, 32, 64, 128, 256, 320, 640, 1280, 2560, 3200]:
    accuracies = []  # Store accuracy for each fold

    # Load data at current sampling rate (32kHz / i)
    data, data_X, data_y = get_data(mounting_tech=mounting_tech, sampling_rate = 32/i )
    
    print("sampling_rate passed:", 32/i)
    print("data_X shape:", data_X.shape)
    print("data_y shape:", data_y.shape)
    print("unique labels:", np.unique(data_y, return_counts=True))  # Check class distribution

    # Perform 5-fold stratified cross-validation
    for train_index, test_index in skf.split(data_X, data_y):
        data_X_train, data_X_test = data_X[train_index], data_X[test_index]
        data_y_train, data_y_test = data_y[train_index], data_y[test_index]
        
        print("\nFold shapes:")
        print("X train:", data_X_train.shape)
        print("X test:", data_X_test.shape)
        print("y train:", data_y_train.shape)
        print("y test:", data_y_test.shape)
        
        # Build and compile the MLP model for current input shape
        model = build_mlp(input_shape=(data_X_train.shape[1],))

        # Train the model
        history = model.fit(
            data_X_train,
            data_y_train,
            epochs=100,
            batch_size=32,
            callbacks=[
                keras.callbacks.ReduceLROnPlateau(monitor = 'loss', factor=0.5,
                      patience=5, min_lr=0.1)
            ],
        )

        # Evaluate model on held-out test set
        eval_result = model.evaluate(data_X_test, data_y_test)
        print(f"\nTest loss: {eval_result[0]}, Test accuracy: {eval_result[1]}")

        print("accuracy: ", eval_result[1])
        accuracies.append(eval_result[1])

        # Generate predictions 
        predictions = model.predict(data_X_test)
        y_pred = np.argmax(predictions, axis=1)

        # Compute and display classification metrics
        from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
        print(classification_report(data_y_test, y_pred))  # Precision, recall, per class
        
    # Compute average accuracy across folds for this sampling rate
    average_accuracy = np.mean(accuracies)
    print(f"MLP Average accuracy for {32/i}KHz: {average_accuracy}")

## CNN + BiLSTM

In [ ]:
# Convolutional Neural Network + Bidirectional LSTM (CNN-BiLSTM) Architecture
# This hybrid architecture combines convolutional layers for local feature extraction
# with bidirectional LSTMs for capturing temporal dependencies in time-series data.
# Architecture design:
# - Multiple Conv1D layers with increasing filters (32, 64, 128, optionally 256, 512, 256, 128, 64)
# - MaxPooling after each conv block to reduce temporal dimension and computational cost
# - Adaptive architecture: Additional conv blocks for longer sequences (>=300 time steps)
# - Bidirectional LSTM (64 units): Captures forward and backward temporal context
# - Dropout (0.5) after LSTM to prevent overfitting
# - Dense output layer with softmax for 4-class classification
# - Optimizer: Adam - Adaptive moment estimation for efficient training
# - Loss: Sparse categorical crossentropy
# This architecture is well-suited for time-series classification tasks like gesture recognition.

import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, Flatten, LSTM, Dense, Bidirectional, Dropout)
import numpy as np

def build_cnnbilstm(input_shape):
    model = Sequential()

    seq_len = input_shape[0]  # Track sequence length for adaptive pooling

    # Helper function to add conv + pool block
    def add_conv_pool(filters):
        nonlocal seq_len
        model.add(Conv1D(filters, kernel_size=3, activation='relu', padding='same'))
        if seq_len >= 2:  # Ensure pooling doesn't reduce below 1
            model.add(MaxPooling1D(pool_size=2))
            seq_len = seq_len // 2

    # Convolutional feature extraction blocks
    add_conv_pool(32)
    add_conv_pool(64)
    add_conv_pool(128)

    # Additional blocks for longer sequences
    if input_shape[0] >= 300:
        add_conv_pool(256)
        add_conv_pool(512)
        add_conv_pool(256)
        add_conv_pool(128)

    add_conv_pool(64)

    # Temporal modeling with bidirectional LSTM
    model.add(Bidirectional(LSTM(64, return_sequences=False)))
    model.add(Dropout(0.5))  # Regularization

    # Classification head
    model.add(Dense(4, activation='softmax'))

    # Compile model
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model

In [ ]:
# CNN-BiLSTM Model Evaluation Across Sampling Rates
# Evaluate the CNN-BiLSTM architecture across different temporal resolutions.
# This model is expected to perform better than MLP due to its ability to capture
# local spatial patterns (via conv) and temporal dependencies (via BiLSTM).
# Same sampling rate range as MLP for fair comparison.

for i in [ 1, 2, 4, 8, 16, 32, 64, 128, 256, 320, 640, 1280, 2560, 3200]:
    accuracies = []

    data, data_X, data_y= get_data(mounting_tech=mounting_tech, sampling_rate = 32/i )
    
    print("sampling_rate passed:", 32/i)
    print("data_X shape:", data_X.shape)
    print("data_y shape:", data_y.shape)
    print("unique labels:", np.unique(data_y, return_counts=True))

    for train_index, test_index in skf.split(data_X, data_y):
        data_X_train, data_X_test = data_X[train_index], data_X[test_index]
        data_y_train, data_y_test = data_y[train_index], data_y[test_index]
        
        print("\nFold shapes:")
        print("X train:", data_X_train.shape)
        print("X test:", data_X_test.shape)
        print("y train:", data_y_train.shape)
        print("y test:", data_y_test.shape)
       
        # Reshape for Conv1D input: (batch, timesteps, features)
        if data_X_train.ndim == 2:
            data_X_train = data_X_train[..., np.newaxis]
            data_X_test = data_X_test[..., np.newaxis]
        model = build_cnnbilstm(input_shape=(data_X_train.shape[1], data_X_train.shape[2]))

        # Train with validation split and early stopping
        # - Validation split 0.2: Monitor generalization performance
        # - Early stopping: Stop if val_loss doesn't improve for 20 epochs, restore best weights
        history = model.fit(
            data_X_train,
            data_y_train,
            validation_split=0.2,
            epochs=100,
            batch_size=32,
            callbacks=[
                keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
            ],
        )

        # Evaluate on test set
        eval_result = model.evaluate(data_X_test, data_y_test)
        print(f"\nTest loss: {eval_result[0]}, Test accuracy: {eval_result[1]}")

        print("accuracy: ", eval_result[1])
        accuracies.append(eval_result[1])

        predictions = model.predict(data_X_test)
        y_pred = np.argmax(predictions, axis=1)

        from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
        print(classification_report(data_y_test, y_pred))
        
    average_accuracy = np.mean(accuracies)
    print(f"CNN-BiLSTM Average accuracy for {32/i}KHz: {average_accuracy}")

## FCN

In [ ]:
# Fully Convolutional Network (FCN) Architecture
# FCNs are designed for time-series classification using only convolutional layers,
# avoiding the need for recurrent connections while maintaining temporal awareness.
# Architecture based on Wang et al. (2017) for time-series classification:
# - Three Conv2D blocks with increasing filters (128, 256, 128)
# - Large kernel sizes (8, 5, 3) to capture multi-scale temporal patterns
# - Batch normalization after each conv for training stability
# - ReLU activation for non-linearity
# - Global average pooling to reduce to fixed-size representation
# - Dense output with softmax
# - Optimizer: Adam for adaptive learning
# - Note: Uses Conv2D with kernel size (k, 1) to effectively implement 1D convolutions
# This architecture is computationally efficient and effective for TSC tasks.

import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, Flatten, LSTM, Dense, Bidirectional, Dropout)
import numpy as np

def build_fcn(input_shape):
    inputs = keras.layers.Input(shape=input_shape)
    
    # First convolutional block
    conv1 = keras.layers.Conv2D(128, (8, 1), strides=(1, 1), padding='same')(inputs)
    conv1 = keras.layers.BatchNormalization()(conv1)
    conv1 = keras.layers.Activation('relu')(conv1)
    
    # Second convolutional block
    conv2 = keras.layers.Conv2D(256, (5, 1), strides=(1, 1), padding='same')(conv1)
    conv2 = keras.layers.BatchNormalization()(conv2)
    conv2 = keras.layers.Activation('relu')(conv2)
    
    # Third convolutional block
    conv3 = keras.layers.Conv2D(128, (3, 1), strides=(1, 1), padding='same')(conv2)
    conv3 = keras.layers.BatchNormalization()(conv3)
    conv3 = keras.layers.Activation('relu')(conv3)
    
    # Global pooling and classification
    full = keras.layers.GlobalAveragePooling2D()(conv3)
    out = keras.layers.Dense(4, activation='softmax')(full)
    
    model = keras.models.Model(inputs=inputs, outputs=out)
    optimizer = keras.optimizers.Adam()  # Adaptive optimizer
    model.compile(loss='sparse_categorical_crossentropy',
                    optimizer=optimizer,
                    metrics=['accuracy'])

    return model

In [ ]:
# FCN Model Evaluation Across Sampling Rates
# Evaluate the Fully Convolutional Network, which uses only convolutional operations
# for efficient processing of time-series data without recurrent components.

for i in [1, 2, 4, 8, 16, 32, 64, 128, 256, 320, 640, 1280, 2560, 3200]:
    accuracies = []

    data, data_X, data_y = get_data(mounting_tech=mounting_tech, sampling_rate = 32/i )

    for train_index, test_index in skf.split(data_X, data_y):
        data_X_train, data_X_test = data_X[train_index], data_X_test = data_X[test_index]
        data_y_train, data_y_test = data_y[train_index], data_y[test_index]
        
        # Reshape for Conv2D input: (batch, timesteps, 1, 1) - treating as 2D with height=1
        data_X_train = data_X_train.reshape((-1, data_X_train.shape[1], 1, 1))
        data_X_test = data_X_test.reshape((-1, data_X_test.shape[1], 1, 1))
        model = build_fcn(input_shape=(data_X_train.shape[1], 1, 1))

        # Train with learning rate scheduling
        # - No validation split (unlike CNN-BiLSTM) as FCN is less prone to overfitting
        # - LR reduction on loss plateau: factor 0.5, patience 5, min LR 0.0001
        history = model.fit(
            data_X_train,
            data_y_train,
            epochs=100,
            batch_size=32,
            callbacks=[
                keras.callbacks.ReduceLROnPlateau(monitor = 'loss', factor=0.5,
                      patience=5, min_lr=0.0001)
            ],
        )

        # Evaluate and collect results
        eval_result = model.evaluate(data_X_test, data_y_test)
        print(f"\nTest loss: {eval_result[0]}, Test accuracy: {eval_result[1]}")

        print("accuracy: ", eval_result[1])
        accuracies.append(eval_result[1])

        predictions = model.predict(data_X_test)
        y_pred = np.argmax(predictions, axis=1)

        from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
        print(classification_report(data_y_test, y_pred))

    average_accuracy = np.mean(accuracies)
    print(f"FCN Average accuracy for {32/i}KHz: {average_accuracy}")

## ResNet

In [ ]:
# Residual Network (ResNet) Architecture for Time-Series Classification
# ResNet addresses vanishing gradients in deep networks through skip connections.
# Architecture consists of residual blocks with:
# - Three conv layers per block (8, 5, 3 kernel sizes) for multi-scale feature extraction
# - Batch normalization for stable training
# - ReLU activation
# - Skip connections that add input to output of conv block
# - Channel expansion in deeper blocks (64 -> 128 -> 128)
# - Global average pooling to fixed-size representation
# - Dense output layer
# - Adam optimizer for training
# This deep architecture can capture hierarchical features while maintaining gradient flow.

import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, Flatten, LSTM, Dense, Bidirectional, Dropout)
import numpy as np

def build_resnet(input_shape, n_feature_maps, nb_classes):
    print ('build conv_x')
    x = keras.layers.Input(shape=(input_shape))
    
    # First residual block
    conv_x = keras.layers.BatchNormalization()(x)
    conv_x = keras.layers.Conv1D(n_feature_maps, 8, 1, padding='same')(conv_x)
    conv_x = keras.layers.BatchNormalization()(conv_x)
    conv_x = keras.layers.Activation('relu')(conv_x)
     
    print ('build conv_y')
    conv_y = keras.layers.Conv1D(n_feature_maps, 5, 1, padding='same')(conv_x)
    conv_y = keras.layers.BatchNormalization()(conv_y)
    conv_y = keras.layers.Activation('relu')(conv_y)
     
    print ('build conv_z')
    conv_z = keras.layers.Conv1D(n_feature_maps, 3, 1, padding='same')(conv_y)
    conv_z = keras.layers.BatchNormalization()(conv_z)
     
    # Skip connection
    is_expand_channels = not (input_shape[-1] == n_feature_maps)
    if is_expand_channels:
        shortcut_y = keras.layers.Conv1D(n_feature_maps, 1, 1,padding='same')(x)
        shortcut_y = keras.layers.BatchNormalization()(shortcut_y)
    else:
        shortcut_y = keras.layers.BatchNormalization()(x)
    print ('Merging skip connection')
    y = keras.layers.Add()([shortcut_y, conv_z])
    y = keras.layers.Activation('relu')(y)
     
    # Second residual block with channel expansion
    print ('build conv_x')
    x1 = y
    conv_x = keras.layers.Conv1D(n_feature_maps*2, 8, 1, padding='same')(x1)
    conv_x = keras.layers.BatchNormalization()(conv_x)
    conv_x = keras.layers.Activation('relu')(conv_x)
     
    print ('build conv_y')
    conv_y = keras.layers.Conv1D(n_feature_maps*2, 5, 1, padding='same')(conv_x)
    conv_y = keras.layers.BatchNormalization()(conv_y)
    conv_y = keras.layers.Activation('relu')(conv_y)
     
    print ('build conv_z')
    conv_z = keras.layers.Conv1D(n_feature_maps*2, 3, 1, padding='same')(conv_y)
    conv_z = keras.layers.BatchNormalization()(conv_z)
     
    # Skip connection
    is_expand_channels = not (input_shape[-1] == n_feature_maps*2)
    if is_expand_channels:
        shortcut_y = keras.layers.Conv1D(n_feature_maps*2, 1, 1,padding='same')(x1)
        shortcut_y = keras.layers.BatchNormalization()(shortcut_y)
    else:
        shortcut_y = keras.layers.BatchNormalization()(x1)
    print ('Merging skip connection')
    y = keras.layers.Add()([shortcut_y, conv_z])
    y = keras.layers.Activation('relu')(y)
     
    # Third residual block (same channels as second)
    print ('build conv_x')
    x1 = y
    conv_x = keras.layers.Conv1D(n_feature_maps*2, 8, 1, padding='same')(x1)
    conv_x = keras.layers.BatchNormalization()(conv_x)
    conv_x = keras.layers.Activation('relu')(conv_x)
     
    print ('build conv_y')
    conv_y = keras.layers.Conv1D(n_feature_maps*2, 5, 1, padding='same')(conv_x)
    conv_y = keras.layers.BatchNormalization()(conv_y)
    conv_y = keras.layers.Activation('relu')(conv_y)
     
    print ('build conv_z')
    conv_z = keras.layers.Conv1D(n_feature_maps*2, 3, 1, padding='same')(conv_y)
    conv_z = keras.layers.BatchNormalization()(conv_z)

    # Skip connection
    is_expand_channels = not (input_shape[-1] == n_feature_maps*2)
    if is_expand_channels:
        shortcut_y = keras.layers.Conv1D(n_feature_maps*2, 1, 1,padding='same')(x1)
        shortcut_y = keras.layers.BatchNormalization()(shortcut_y)
    else:
        shortcut_y = keras.layers.BatchNormalization()(x1)
    print ('Merging skip connection')
    y = keras.layers.Add()([shortcut_y, conv_z])
    y = keras.layers.Activation('relu')(y)
     
    # Global pooling and classification
    full = keras.layers.GlobalAveragePooling1D()(y)
    out = keras.layers.Dense(nb_classes, activation='softmax')(full)
    print ('        -- model was built.')

    model = keras.models.Model(inputs=x, outputs=out)
    optimizer = keras.optimizers.Adam()
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])
    
    return model

In [ ]:
# ResNet Model Evaluation Across Sampling Rates
# Evaluate the deep residual network, which should excel at capturing
# hierarchical features through skip connections.

for i in [1, 2, 4, 8, 16, 32, 64, 128, 256, 320, 640, 1280, 2560, 3200]:
    accuracies = []

    data, data_X, data_y = get_data(mounting_tech=mounting_tech, sampling_rate = 32/i )

    for train_index, test_index in skf.split(data_X, data_y):
        data_X_train, data_X_test = data_X[train_index], data_X[test_index]
        data_y_train, data_y_test = data_y[train_index], data_y[test_index]
        
        # Reshape for Conv1D: add channel dimension
        if data_X_train.ndim == 2:
            data_X_train = data_X_train[..., np.newaxis]
            data_X_test = data_X_test[..., np.newaxis]

        # Build ResNet with 64 base feature maps, 4 output classes
        model = build_resnet(
            input_shape=(data_X_train.shape[1], data_X_train.shape[2]),
            n_feature_maps=64,
            nb_classes=4
        )

        # Train with LR scheduling
        # - Similar to FCN: LR reduction on loss plateau
        history = model.fit(
            data_X_train,
            data_y_train,
            epochs=100,
            batch_size=32,
            callbacks=[
                keras.callbacks.ReduceLROnPlateau(monitor = 'loss', factor=0.5,
                      patience=5, min_lr=0.0001)
            ],
        )

        # Evaluate
        eval_result = model.evaluate(data_X_test, data_y_test)
        print(f"\nTest loss: {eval_result[0]}, Test accuracy: {eval_result[1]}")

        print("accuracy: ", eval_result[1])
        accuracies.append(eval_result[1])

        predictions = model.predict(data_X_test)
        y_pred = np.argmax(predictions, axis=1)

        from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
        print(classification_report(data_y_test, y_pred))

    average_accuracy = np.mean(accuracies)
    print(f"ResNet Average accuracy for {32/i}KHz: {average_accuracy}")

## SVM

In [ ]:
# Support Vector Machine (SVM) Evaluation
# SVM serves as a traditional machine learning baseline for comparison with deep learning models.
# We evaluate only at the lowest sampling rate (0.01kHz) as SVMs are less affected by
# high-dimensional data and can be computationally expensive on full-resolution signals.
# Default SVM parameters are used (RBF kernel, C=1.0) as a standard baseline.

for i in [1, 2, 4, 8, 16, 32, 64, 128, 256, 320, 640, 1280, 2560, 3200]:
    accuracies = []

    data, data_X, data_y= get_data(mounting_tech=mounting_tech, sampling_rate = 32/i )

    for train_index, test_index in skf.split(data_X, data_y):
        data_X_train, data_X_test = data_X[train_index], data_X[test_index]
        data_y_train, data_y_test = data_y[train_index], data_y[test_index]
    
        from sklearn import svm
        from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

        # Create SVM classifier with default parameters
        classifier = svm.SVC()

        # Train the model
        classifier.fit(data_X_train, data_y_train)
        
        # Predict and evaluate
        y_pred = classifier.predict(data_X_test)
        acc = accuracy_score(data_y_test, y_pred)
        
        # Display detailed classification metrics
        print(classification_report(data_y_test, y_pred))

        accuracies.append(acc)
        print(acc)

    print("Mean accuracy:", np.mean(accuracies))

## Random Forest

In [ ]:
# Random Forest Evaluation
# Random Forest provides another traditional ML baseline, using an ensemble of decision trees.
# Like SVM, evaluated only at lowest sampling rate for efficiency.
# Number of estimators set to 120, which provides good performance without excessive computation.
# Random Forest is robust to overfitting and can handle high-dimensional data well.

for i in [1, 2, 4, 8, 16, 32, 64, 128, 256, 320, 640, 1280, 2560, 3200]:
    accuracies = []

    data, data_X, data_y= get_data(mounting_tech=mounting_tech, sampling_rate = 32/i )

    for train_index, test_index in skf.split(data_X, data_y):
        data_X_train, data_X_test = data_X[train_index], data_X_test = data_X[test_index]
        data_y_train, data_y_test = data_y[train_index], data_y[test_index]
    
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

        # Create Random Forest with 120 trees
        classifier = RandomForestClassifier(n_estimators=120)

        # Train the model
        classifier.fit(data_X_train, data_y_train)
        
        # Predict and evaluate
        y_pred = classifier.predict(data_X_test)
        acc = accuracy_score(data_y_test, y_pred)
        
        # Display detailed metrics
        print(classification_report(data_y_test, y_pred))

        accuracies.append(acc)
        print(acc)

    print("Mean accuracy:", np.mean(accuracies))